# ALS-Finder: Big-Data LiDAR Tiling & Single-Tile Streaming Tutorial

Welcome to **ALS-Finder**! This interactive tutorial demonstrates how to search, grid, and stream massive airborne LiDAR point cloud datasets on-demand without downloading gigabytes of unneeded raw files upfront.

---

### 🌟 Key Concepts & Architectural Components

| Concept | Description | Underlying Module |
| :--- | :--- | :--- |
| **`workspace_dir`** | Isolated root directory containing catalog metadata, grids, and outputs | Filesystem path |
| **`manifest.json`** | Catalog database indexing federated remote cloud endpoints (USGS, NOAA, OpenTopography) | `als_finder.cli search` |
| **`grid.gpkg`** | Spatial index dividing ROI into uniform metric core tiles + edge overlap buffers | `als_finder.core.grid_manager` |
| **`get_tile_spec()`** | Zero-copy query to retrieve single-tile bounds, crop strings, and Hive paths | `als_finder.core.grid_manager` |
| **`stream_single_tile()`** | On-demand HTTP streaming of a single buffered tile with SMRF and HAG normalization | `als_finder.core.standardization` |

--- 
## Step 1: Environment & Workspace Initialization

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely.geometry import box

# Ensure als_finder module is discoverable on python path
repo_root = Path("..").resolve() if Path("../src").exists() else Path(".").resolve()
sys.path.insert(0, str(repo_root / "src"))

from als_finder.core.grid_manager import (
    build_workspace_grid,
    get_tile_spec,
    create_tile_grid_index,
)
from als_finder.core.standardization import stream_single_tile

# Setup a demo workspace directory
workspace_dir = Path("./tahoe_demo_workspace").resolve()
workspace_dir.mkdir(parents=True, exist_ok=True)

print(f"✓ Python interpreter: {sys.executable}")
print(f"✓ Workspace directory ready: {workspace_dir}")

--- 
## Step 2: Search Remote LiDAR Catalogs (Zero Point Cloud Download)

We query federated providers (USGS 3DEP EPT / NOAA / OpenTopography) across a Lake Tahoe Region of Interest (ROI). This creates `catalog/manifest.json` and `catalog/catalog.gpkg` without fetching any point clouds.

In [ ]:
# Lake Tahoe sample bounding box (min_lon, min_lat, max_lon, max_lat)
tahoe_roi = "-120.08,38.91,-120.06,38.93"

cmd = [
    sys.executable, "-m", "als_finder.cli", "search",
    "--roi", tahoe_roi,
    "--workspace", str(workspace_dir),
    "--provider", "USGS_EPT,NOAA_STAC"
]

print(f"Searching catalogs for ROI: {tahoe_roi}...")
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print("Logs:", res.stderr)

manifest_path = workspace_dir / "catalog" / "manifest.json"
assert manifest_path.exists(), "Search stage failed to produce manifest.json!"

with open(manifest_path) as f:
    datasets = json.load(f)
print(f"✓ Successfully indexed {len(datasets)} dataset(s) in catalog!")

--- 
## Step 3: Lazy Grid Generation (Core + Buffer Tiling)

The user specifies their target **Core Tile Size** (e.g. `1200m`) and **Overlap Buffer** (e.g. `30m`).

`get_tile_spec()` automatically:
1. Discovers that the grid does not exist yet,
2. Snaps the ROI bounds to metric multiples in the projected CRS (e.g. UTM),
3. Generates the spatial vector grid and exports it into Hive storage: `catalog/grids/tilesize=1200/buffer=30/grid.gpkg`.

In [ ]:
# Request Tile ID #0 with 1200m core size + 30m buffer
tile_spec = get_tile_spec(workspace_dir, tile_id=0, tile_size=1200, buffer_size=30)

print("=" * 65)
print("                SINGLE TILE SPECIFICATION (TILE #0)              ")
print("=" * 65)
print(f"Tile ID:               {tile_spec['tile_id']}")
print(f"Basename:              {tile_spec['basename']}")
print(f"Projected Grid CRS:    {tile_spec['grid_crs']}")
print(f"Proposed Hive Path:    {tile_spec['hive_path']}")
print(f"Core PDAL Crop Bounds: {tile_spec['core_bounds_str']}")
print(f"Buffered Stream Bounds:{tile_spec['buffered_bounds_str']}")
print("=" * 65)

--- 
## Step 4: Visualize Spatial Grid & Overlapping Datasets in Python

In [ ]:
grid_gpkg = workspace_dir / "catalog" / "grids" / "tilesize=1200" / "buffer=30" / "grid.gpkg"
grid_gdf = gpd.read_file(grid_gpkg)

print(f"Total Tiles in Projected Grid: {len(grid_gdf)}")
display(grid_gdf.head())

--- 
## Step 5: On-Demand Single-Tile Streaming (`stream_single_tile`)

Now we stream **only Tile #0** over HTTP. Under the hood, this executes:
- `readers.ept` with bounding box constraint matching `buffered_bounds_str`,
- SMRF ground classification & HAG (Height Above Ground) tree height normalization,
- Direct output of standardized `.laz` tile inside an OS-level memory sandbox.

In [ ]:
out_tile_path = workspace_dir / "output_tiles" / tile_spec["basename"]
out_tile_path.parent.mkdir(parents=True, exist_ok=True)

print(f"Streaming single tile #{tile_spec['tile_id']} over HTTP...")
streamed_path = stream_single_tile(
    manifest_or_grid_path=manifest_path,
    tile_id=0,
    out_path=out_tile_path,
    buffer_size=30,
    crs=tile_spec["grid_crs"],
    overwrite=True
)

size_mb = streamed_path.stat().st_size / (1024 * 1024)
print(f"✓ Successfully streamed and standardized single tile!")
print(f"  Location: {streamed_path}")
print(f"  Size:     {size_mb:.2f} MB")

--- 
## Step 6: CLI Equivalent for HPC Job Arrays

On high-performance compute clusters (e.g. SDSC Expanse), Slurm jobs execute this same single-tile stream via the CLI using `$SLURM_ARRAY_TASK_ID`:

```bash
# Slurm Worker single-tile execution command:
als-finder fetch-tile \
  --manifest workspace/catalog/manifest.json \
  --tile-id $SLURM_ARRAY_TASK_ID \
  --buffer-size 30 \
  --crs EPSG:32610 \
  --output workspace/scratch/tile_${SLURM_ARRAY_TASK_ID}.laz
```

In [ ]:
# Execute the CLI fetch-tile command to verify parity
cli_out = workspace_dir / "output_tiles" / "tile_cli_test.laz"
fetch_cmd = [
    sys.executable, "-m", "als_finder.cli", "fetch-tile",
    "--manifest", str(manifest_path),
    "--tile-id", "0",
    "--buffer-size", "30",
    "--output", str(cli_out),
    "--json"
]

res = subprocess.run(fetch_cmd, capture_output=True, text=True)
print("CLI JSON Response:")
print(res.stdout)

--- 
## Summary

You have successfully demonstrated the end-to-end big-data streaming workflow:
1. **Search & Discovery**: Lightweight metadata query producing `manifest.json`.
2. **Dynamic Gridding**: ROI decomposed into uniform core tiles + edge buffers.
3. **On-Demand Streaming**: Single tiles fetched over HTTP with on-the-fly SMRF and HAG standardization.
4. **HPC Ready**: Zero-disk master node footprint, completely decoupled for Slurm array workers.